# Saudi Labor Law — Multilingual RAG Agent

This Colab notebook builds a RAG agent that:
- accepts Arabic or English questions
- retrieves evidence from the uploaded PDF
- uses GPT-5.6 Sol for the final answer
- refuses to invent information when the PDF does not support the answer
- returns the retrieved source pages/excerpts for verification


In [1]:
!pip -q install -U langchain langchain-community langchain-openai langgraph faiss-cpu pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
from getpass import getpass

OPENAI_API_KEY = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Optional: LangSmith tracing
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API key: ")
# os.environ["LANGSMITH_PROJECT"] = "saudi-labor-law-rag"

print("API key loaded.")


Enter your OpenAI API key: ··········
API key loaded.


In [3]:
# BASE DATA

from google.colab import files

uploaded = files.upload()
PDF_PATH = next(iter(uploaded.keys()))
print("Using:", PDF_PATH)


Saving labor-law.pdf to labor-law.pdf
Saving أطر العمل التنظيمية للائحة التنفيذية للموارد البشرية.pdf to أطر العمل التنظيمية للائحة التنفيذية للموارد البشرية.pdf
Saving الدليل الإجرائي لتنظيم العمل المرن.pdf to الدليل الإجرائي لتنظيم العمل المرن.pdf
Saving اللائحة التنفيذية لنظام العمل وملحقاتها (1).pdf to اللائحة التنفيذية لنظام العمل وملحقاتها (1).pdf
Saving اللائحة التنفيذية لنظام العمل وملحقاتها.pdf to اللائحة التنفيذية لنظام العمل وملحقاتها.pdf
Using: labor-law.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"PDF pages loaded: {len(pages)}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=250,
    separators=[
        "\nالمادة ",
        "\nArticle ",
        "\nالباب ",
        "\nالفصل ",
        "\n\n",
        "\n",
        " ",
        ""
    ],
)

docs = splitter.split_documents(pages)

# Normalize metadata so citations are easy to use later.
for i, doc in enumerate(docs):
    page = doc.metadata.get("page")
    doc.metadata["page_number"] = (page + 1) if isinstance(page, int) else None
    doc.metadata["source_file"] = PDF_PATH
    doc.metadata["chunk_id"] = i

print(f"Chunks created: {len(docs)}")


/tmp/ipykernel_1688/3706689381.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = FAISS.from_documents(docs, embeddings)

# MMR gives diverse relevant chunks instead of near-duplicates.
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 20,
        "lambda_mult": 0.4,
    },
)

print("Vector store is ready.")


Vector store is ready.


In [ ]:
from langchain_core.tools import tool

last_sources = []

@tool
def retrieve_labor_law(query: str) -> str:
    """
    Retrieve the most relevant evidence from the Saudi Labor Law PDF.
    """

    global last_sources

    # Retrieve more candidates first
    results = retriever.invoke(query)

    # Remove duplicate pages/chunks
    unique_results = []
    seen = set()

    for doc in results:
        page = doc.metadata.get("page_number")
        content = doc.page_content.strip()

        key = (page, content[:150])

        if key not in seen:
            seen.add(key)
            unique_results.append(doc)

    # Keep only the best 4
    unique_results = unique_results[:4]

    last_sources = []

    evidence_blocks = []

    for i, doc in enumerate(unique_results, start=1):

        source_id = f"S{i}"

        page = doc.metadata.get("page_number")
        content = doc.page_content.strip()

        source = {
            "id": source_id,
            "file": PDF_PATH,
            "page": page,
            "content": content
        }

        last_sources.append(source)

        evidence_blocks.append(
            f"""
[{source_id}]
Page: {page}

Evidence:
{content}
"""
        )

    if not evidence_blocks:
        return "NO_RELEVANT_EVIDENCE_FOUND"

    return "\n".join(evidence_blocks)

In [ ]:
''' from langchain_core.tools import tool

# Keeps the exact evidence returned by the retriever so the notebook can
# print verified source information after the agent finishes.
last_sources = []

@tool
def retrieve_labor_law(query: str) -> str:
    """Retrieve authoritative evidence from the uploaded Saudi Labor Law PDF."""
    global last_sources

    results = retriever.invoke(query)
    last_sources = []
    blocks = []

    for idx, doc in enumerate(results, start=1):
        source_id = f"S{idx}"
        page = doc.metadata.get("page_number")
        source_file = doc.metadata.get("source_file", PDF_PATH)
        content = doc.page_content.strip()

        item = {
            "id": source_id,
            "file": source_file,
            "page": page,
            "content": content,
        }
        last_sources.append(item)

        blocks.append(
            f"[{source_id}] FILE: {source_file} | PAGE: {page}\n"
            f"EVIDENCE:\n{content}"
        )

    if not blocks:
        return "NO_RELEVANT_EVIDENCE_FOUND"

    return "\n\n".join(blocks)

print("Retrieval tool ready.")'''


Retrieval tool ready.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI(
    model="gpt-5.6-sol",
    temperature=0,
)

SYSTEM_PROMPT = """
You are a document-grounded RAG assistant for the uploaded Saudi Labor Law PDF.

Your job is to answer questions ONLY using evidence retrieved from the PDF.

========================
RETRIEVAL RULES
========================

1. For every Labor Law question, ALWAYS call retrieve_labor_law first.

2. Use only the retrieved evidence.

3. Do NOT use your general knowledge to fill missing information.

4. If the retrieved evidence is insufficient, say:
   Arabic:
   "لم أجد معلومات كافية في الوثيقة المرفوعة للإجابة عن هذا السؤال."

   English:
   "I could not find enough information in the uploaded document to answer this question."

5. Never invent:
   - article numbers
   - page numbers
   - dates
   - percentages
   - penalties
   - exceptions
   - legal conditions

========================
ANSWER FORMAT
========================

Answer in the same language as the question.

For Arabic questions use exactly this structure:

## الإجابة

[Clear and concise answer]

## المصدر

📄 **الوثيقة:** [file name]

📍 **الصفحة:** [page number]

📌 **الدليل:**
> [short relevant evidence from the retrieved source]

For English questions use:

## Answer

[Clear and concise answer]

## Source

📄 **Document:** [file name]

📍 **Page:** [page number]

📌 **Evidence:**
> [short relevant evidence from the retrieved source]

========================
SOURCE RULES
========================

6. Only cite source IDs that actually exist in the retrieved evidence.

7. Prefer the MOST RELEVANT source instead of listing every retrieved chunk.

8. If multiple sources directly support the answer, list only the relevant sources.

9. Do not show irrelevant retrieved chunks to the user.

10. The source page number must come directly from the retrieval tool.

11. Keep the evidence quote short and directly relevant.

12. Do not expose internal tool calls, reasoning, JSON, embeddings, vector database information, or retrieval debugging information.

========================
LEGAL SAFETY
========================

13. If the document contains ambiguity or conflicting provisions, clearly state that.

14. Do not claim that information is current law unless the uploaded document itself supports that conclusion.

15. This assistant provides document-grounded information and is not a substitute for professional legal advice.
"""

agent = create_agent(
    model=llm,
    tools=[retrieve_labor_law],
    system_prompt=SYSTEM_PROMPT,
)

print("Agent ready.")


Agent ready.


In [ ]:
def ask(question: str):

    global last_sources

    last_sources = []

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    })

    messages = result.get("messages", [])

    final_message = messages[-1]

    answer = getattr(
        final_message,
        "content",
        str(final_message)
    )

    # بعض النماذج ترجع content كـ list
    if isinstance(answer, list):

        text_parts = []

        for item in answer:

            if isinstance(item, dict):

                if item.get("type") == "text":
                    text_parts.append(
                        item.get("text", "")
                    )

            else:
                text_parts.append(str(item))

        answer = "\n".join(text_parts)

    print(answer)

    return result

    messages = result.get("messages", [])
    final_message = messages[-1]
    answer = getattr(final_message, "content", str(final_message))

    print("=" * 80)
    print("ANSWER")
    print("=" * 80)
    print(answer)

    print("\n" + "=" * 80)
    print("VERIFIED RETRIEVED SOURCES")
    print("=" * 80)

    if not last_sources:
        print("No retrieved sources were recorded.")
        return result

    for src in last_sources:
        print(f"[{src['id']}] {src['file']} — page {src['page']}")
        print(src["content"][:700].replace("\n", " "))
        print("-" * 80)

    return result


## Test 1 — Arabic

Try a question such as:

`ما هي ساعات العمل اليومية والأسبوعية حسب نظام العمل؟`


In [ ]:
ask("ما هي ساعات العمل اليومية والأسبوعية حسب نظام العمل؟")


## الإجابة

ساعات العمل النظامية هي **ثماني ساعات يومياً** أو **ثمانٍ وأربعون ساعة أسبوعياً**.

## المصدر

📄 **الوثيقة:** نظام العمل السعودي المرفوع

📍 **الصفحة:** 33

📌 **الدليل:**
> «ثماني ساعات يومياً أو ثمان وأربعين ساعة أسبوعياً.»


{'messages': [HumanMessage(content='ما هي ساعات العمل اليومية والأسبوعية حسب نظام العمل؟', additional_kwargs={}, response_metadata={}, id='4c0c5329-07cc-4ed4-a65d-4e4f44d03ec2'),
  AIMessage(content=[{'arguments': '{"query":"ساعات العمل اليومية والأسبوعية الحد الأقصى ثماني ساعات يومياً ثمان وأربعون ساعة أسبوعياً وتخفيض رمضان"}', 'call_id': 'call_mhDO1stPHR6UnJp7toPFIU2b', 'name': 'retrieve_labor_law', 'type': 'function_call', 'id': 'fc_0ffe31cdffd33535006aa267dba50487d2beaf6d9ade7224b5', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0ffe31cdffd33535006aa267dad42087d2986aedaf3c79a1a5', 'created_at': 1789028314.0, 'metadata': {}, 'model': 'gpt-5.6-sol', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.6-sol'}, id='resp_0ffe31cdffd33535006aa267dad42087d2986aedaf3c79a1a5', tool_calls=[{'name': 'retrieve_labor_law', 'args': {'query': 'ساعات العمل اليومية والأسبوعية الحد الأقصى ثماني ساعات

In [ ]:
ask("ابي اصير بحار ايش الشروط؟")


## الإجابة

يشترط للعمل بحّارًا:

1. أن يكون عمرك **18 سنة فأكثر**.
2. أن تحمل **شهادة تتيح لك العمل في الخدمة البحرية**.
3. أن تكون **لائقًا طبيًا**.

## المصدر

📄 **الوثيقة:** نظام العمل السعودي المرفوع

📍 **الصفحة:** 52

📌 **الدليل:**
> «يشترط فيمن يعمل بحاراً: أن يكون قد أتم من العمر ثماني عشرة سنة، وأن يكون حاصلاً على شهادة تتيح له العمل في الخدمة البحرية، وأن يكون لائقاً طبياً.»


{'messages': [HumanMessage(content='ابي اصير بحار ايش الشروط؟', additional_kwargs={}, response_metadata={}, id='9cc6c5c9-0003-48da-8a5a-5ea60001884b'),
  AIMessage(content=[{'id': 'rs_0555549f6a4f7694006aa2685fd38887d290b156734675161e', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqomhixVMoBNeiabAY_CYHkI-Dn3mkzMibXKhMCRg9Px0fW8D5Ue8J1AU0KiFY0TOl8zVVBIGUSBtREAXo1vni_zDWy9ydkXnYnGT4gqsstqO6PY0fOz296Eyb7MuoccdDAeCEm16gV8iJzPhqqxjLSL-SpJI7X5SPR_514d7YxcOLbOUsa7MEzERM6UUJkYzNHPH2fD2sVN9NXkDGjdpw_QNqxm_badKwo5u5SmADeduLo0zR0fCIU8NwTE1sBB2q2sBV_lirZY59xGCCaog10VMDUIO5Y3P4ygnC6S0dfemy2DkqjtzUoCZDkPSR4z-gHLTO9YF0MP2Pmsp9_D0aTCwGlRjYZva3f5VlHm9lNRcm48dKLzIHZMZTuoIEnqfnCBTnNyWLtSL6D2wK2n1RPjOrtQu9Rwf-sg-OnBVgyvw_kSA1oDl881LVwFgxEGooLtQ8_BC15sgFxQCwzyIQwryZRgHmijAKm9sLN8RBH0MQt2_VChqbR-cBPK-wvSYak4ZgFVuIEaHDsY1dV8FAiTTsg2_LxJ0t7PKjzez5ZV_zQWahlqRmE6-RQcRP9zcLLk2RnfaW6m4nXAJmgigoLRz4NLpKl8LNnDIgt-9kBCwvriLf3afDg8oJKC9xTH1Ygu6iah-uWYeEV0jSXGbsBQZwvknJf4bh__sMfBLOda

## Test 2 — English

`What are the normal working hours under the Labor Law?`


In [ ]:
ask("What are the normal working hours under the Labor Law?")


## Test 3 — Out-of-document question

This checks whether the agent refuses to invent an answer when the PDF does not contain enough evidence.


In [ ]:
ask("Who is the current CEO of Apple?")


## Interactive mode

Run this cell and keep asking Arabic or English questions.


In [ ]:
while True:
    q = input("\nYour question (type exit to stop): ")
    if q.strip().lower() in {"exit", "quit", "خروج"}:
        print("Stopped.")
        break
    ask(q)
